# Part 0 - Loads and Helper Functions

In [18]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
import lightgbm as lgb

### Global Configurations

In [19]:
# Reproducibility
SEED = 42

# Paths
TRAIN_PATH = "../data/training_set_VU_DM.csv"
TEST_PATH = "../data/test_set_VU_DM.csv"

# Target engineering
BOOKING_RELEVANCE = 5
CLICK_RELEVANCE = 1
TARGET_COL = "target"

# Validation split
TEST_SIZE = 0.2

# Ranking evaluation
NDCG_AT = 5

# LightGBM baseline parameters
LGB_PARAMS = {
    "objective": "lambdarank",
    "metric": "ndcg",
    "ndcg_eval_at": [NDCG_AT],

    "learning_rate": 0.05,
    "n_estimators": 200,
    "num_leaves": 31,

    "random_state": SEED,
    "n_jobs": -1,
}

# Columns to remove from features
LEAKAGE_COLS = [
    "target",
    "booking_bool",
    "click_bool",
    "gross_bookings_usd",
    "position",
]

ID_COLS = [
    "srch_id",
    "site_id",
    "visitor_location_country_id",
    "prop_country_id",
    "prop_id",
    "srch_destination_id",
]

DATE_COLS = [
    "date_time",
    "date",
]

DROP_COLS = LEAKAGE_COLS + ID_COLS + DATE_COLS

### Defining Columns and Data Types

In [20]:
cont_cols = [
    'visitor_hist_starrating',
    'visitor_hist_adr_usd',
    'prop_review_score',
    'prop_location_score1',
    'prop_location_score2',
    'prop_log_historical_price',
    'price_usd',
    'srch_query_affinity_score',
    'orig_destination_distance',
    'gross_bookings_usd',
    'comp1_rate_percent_diff',
    'comp2_rate_percent_diff',
    'comp3_rate_percent_diff',
    'comp4_rate_percent_diff',
    'comp5_rate_percent_diff',
    'comp6_rate_percent_diff',
    'comp7_rate_percent_diff',
    'comp8_rate_percent_diff'
]

bool_cols = [
    'prop_brand_bool',
    'promotion_flag',
    'srch_saturday_night_bool',
    'random_bool',
    'click_bool',
    'booking_bool'
]

id_cols = [
    'srch_id',
    'site_id',
    'visitor_location_country_id',
    'prop_country_id',
    'prop_id',
    'srch_destination_id'
]

cat_cols = [
    'comp1_rate', 'comp1_inv',
    'comp2_rate', 'comp2_inv',
    'comp3_rate', 'comp3_inv',
    'comp4_rate', 'comp4_inv',
    'comp5_rate', 'comp5_inv',
    'comp6_rate', 'comp6_inv',
    'comp7_rate', 'comp7_inv',
    'comp8_rate', 'comp8_inv'
]

int_cols = [
    'prop_starrating',
    'position',
    'srch_length_of_stay',
    'srch_booking_window',
    'srch_adults_count',
    'srch_children_count',
    'srch_room_count'
]

In [21]:
dtype_dict = {}

# float columns
for col in cont_cols:
    dtype_dict[col] = 'float32'

# bool columns
for col in bool_cols:
    dtype_dict[col] = 'bool'

# id columns
for col in id_cols:
    dtype_dict[col] = 'int32'

# categorical columns
for col in cat_cols:
    dtype_dict[col] = 'category'

# small integer columns
for col in int_cols:
    dtype_dict[col] = 'int8'

### Helper Functions

In [22]:
# Load Training Data
def load_train_data(
    path,
    booking_relevance=5,
    click_relevance=1
):

    df = pd.read_csv(
        path,
        dtype=dtype_dict,
        low_memory=False
    )

    # Create Target
    df["target"] = (
        booking_relevance * df["booking_bool"].astype("int8") +
        click_relevance * df["click_bool"].astype("int8")
    ).astype("int8")

    return df

In [23]:
# Load Testing Data
def load_test_data(path):

    df = pd.read_csv(
        path,
        dtype=dtype_dict,
        low_memory=False
    )

    return df

In [24]:
# Stateless Prepocess Function (applied before Train-Val split)
def stateless_preprocess(
    df,
    comp_rate_diff_upper_limit=150
):

    df = df.copy()

    # ==================================================
    # DATE PARSING
    # ==================================================

    df["date_time"] = pd.to_datetime(df["date_time"])

    # ==================================================
    # WRONG VALUE HANDLING
    # ==================================================

    # Negative booking windows are invalid
    df["booking_window_was_negative"] = (
        df["srch_booking_window"] < 0
    )

    df["srch_booking_window"] = (
        df["srch_booking_window"]
        .clip(lower=0)
    )

    # ==================================================
    # OUTLIER CAPPING
    # ==================================================

    # Cap extreme competitor rate differences
    for i in range(1, 9):

        col = f"comp{i}_rate_percent_diff"

        df[col] = (
            df[col]
            .clip(upper=comp_rate_diff_upper_limit)
        )

    # ==================================================
    # SEMANTIC MISSING VALUES -> NaN
    # ==================================================

    # Unknown starrating
    df["prop_starrating"] = (
        df["prop_starrating"]
        .replace(0, np.nan)
    )

    # Unknown historical price
    df["prop_log_historical_price"] = (
        df["prop_log_historical_price"]
        .replace(0, np.nan)
    )

    # Unknown review score
    df["prop_review_score"] = (
        df["prop_review_score"]
        .replace(0, np.nan)
    )

    # ==================================================
    # MISSING VALUE FLAG FEATURES
    # ==================================================

    # Missing starrating
    df["no_starrating"] = (
        df["prop_starrating"].isna()
    )

    # Missing historical price
    df["prop_no_sold"] = (
        df["prop_log_historical_price"].isna()
    )

    # Missing review score
    df["no_review"] = (
        df["prop_review_score"].isna()
    )

    # Missing distance
    df["missing_distance"] = (
        df["orig_destination_distance"].isna()
    )

    # Missing affinity score
    df["no_affinity"] = (
        df["srch_query_affinity_score"].isna()
    )

    # Missing visitor history
    df["no_vis_hist"] = (
        df["visitor_hist_starrating"].isna()
    )

    # ==================================================
    # COMPETITOR AVAILABILITY FLAGS
    # ==================================================

    for i in range(1, 9):

        diff_col = f"comp{i}_rate_percent_diff"

        df[f"has_comp{i}_diff"] = (
            df[diff_col].notna()
        )

    # ==================================================
    # COMPETITOR CATEGORICAL MISSING VALUES
    # ==================================================

    # Missing competitor info gets explicit category
    for i in range(1, 9):

        rate_col = f"comp{i}_rate"
        inv_col = f"comp{i}_inv"

        df[rate_col] = (
            df[rate_col]
            .astype("float32")
            .fillna(99)
            .astype("category")
        )

        df[inv_col] = (
            df[inv_col]
            .astype("float32")
            .fillna(99)
            .astype("category")
        )

    return df

In [25]:
# Train-Val Split Function
def train_val_split(
    df,
    group_col="srch_id",
    test_size=0.2,
    random_state=42
):

    splitter = GroupShuffleSplit(
        test_size=test_size,
        n_splits=1,
        random_state=random_state
    )

    train_idx, val_idx = next(
        splitter.split(
            df,
            groups=df[group_col]
        )
    )

    train_df = df.iloc[train_idx].copy()
    val_df = df.iloc[val_idx].copy()

    return train_df, val_df

In [26]:
# Fit Function (get global statistics from Train Set only)
def fit_learned_preprocessing(
    train_df,
    price_usd_quantile=0.995
):

    stats = {}

    # ==================================================
    # PRICE USD QUANTILE CAP
    # ==================================================

    stats["price_usd_upper_cap"] = (
        train_df["price_usd"]
        .quantile(price_usd_quantile)
    )

    # ==================================================
    # PROP STARRATING
    # ==================================================

    stats["prop_starrating_mode"] = (
        train_df["prop_starrating"]
        .mode()[0]
    )

    # ==================================================
    # PROP LOG HISTORICAL PRICE
    # ==================================================

    stats["prop_log_historical_price_median"] = (
        train_df["prop_log_historical_price"]
        .median()
    )

    # ==================================================
    # PROP REVIEW SCORE
    # ==================================================

    stats["prop_review_score_hotel_medians"] = (
        train_df
        .groupby(
            "prop_id",
            observed=True
        )["prop_review_score"]
        .median()
    )

    stats["prop_review_score_global_median"] = (
        train_df["prop_review_score"]
        .median()
    )

    # ==================================================
    # PROP LOCATION SCORE 2
    # ==================================================

    stats["prop_location_score2_hotel_medians"] = (
        train_df
        .groupby(
            "prop_id",
            observed=True
        )["prop_location_score2"]
        .median()
    )

    stats["prop_location_score2_global_median"] = (
        train_df["prop_location_score2"]
        .median()
    )

    # ==================================================
    # ORIG DESTINATION DISTANCE
    # ==================================================

    stats["orig_destination_distance_median"] = (
        train_df["orig_destination_distance"]
        .median()
    )

    # ==================================================
    # SEARCH QUERY AFFINITY SCORE
    # ==================================================

    stats["srch_query_affinity_score_median"] = (
        train_df["srch_query_affinity_score"]
        .median()
    )

    # ==================================================
    # VISITOR HISTORY FEATURES
    # ==================================================

    stats["visitor_hist_starrating_median"] = (
        train_df["visitor_hist_starrating"]
        .median()
    )

    stats["visitor_hist_adr_usd_median"] = (
        train_df["visitor_hist_adr_usd"]
        .median()
    )

    # ==================================================
    # COMP RATE PERCENT DIFF FEATURES
    # ==================================================

    for i in range(1, 9):

        col = f"comp{i}_rate_percent_diff"

        stats[f"{col}_median"] = (
            train_df[col]
            .median()
        )

    return stats

In [27]:
# Transformation Function - applied to all TRAIN, VAL, TEST sets based on Training Stats
def apply_learned_preprocessing(
    df,
    stats
):

    df = df.copy()

    # ==================================================
    # PRICE USD
    # ==================================================

    df["price_usd"] = (
        df["price_usd"]
        .clip(upper=stats["price_usd_upper_cap"])
        .astype("float32")
    )

    # ==================================================
    # PROP STARRATING
    # ==================================================

    df["prop_starrating"] = (
        df["prop_starrating"]
        .fillna(stats["prop_starrating_mode"])
        .astype("int8")
    )

    # ==================================================
    # PROP LOG HISTORICAL PRICE
    # ==================================================

    df["prop_log_historical_price"] = (
        df["prop_log_historical_price"]
        .fillna(
            stats["prop_log_historical_price_median"]
        )
        .astype("float32")
    )

    # ==================================================
    # PROP REVIEW SCORE
    # ==================================================

    hotel_review_medians = (
        df["prop_id"]
        .map(
            stats["prop_review_score_hotel_medians"]
        )
    )

    df["prop_review_score"] = (
        df["prop_review_score"]
        .fillna(hotel_review_medians)
        .fillna(
            stats["prop_review_score_global_median"]
        )
        .astype("float32")
    )

    # ==================================================
    # PROP LOCATION SCORE 2
    # ==================================================

    hotel_location_medians = (
        df["prop_id"]
        .map(
            stats[
                "prop_location_score2_hotel_medians"
            ]
        )
    )

    df["prop_location_score2"] = (
        df["prop_location_score2"]
        .fillna(hotel_location_medians)
        .fillna(
            stats[
                "prop_location_score2_global_median"
            ]
        )
        .astype("float32")
    )

    # ==================================================
    # ORIG DESTINATION DISTANCE
    # ==================================================

    df["orig_destination_distance"] = (
        df["orig_destination_distance"]
        .fillna(
            stats[
                "orig_destination_distance_median"
            ]
        )
        .astype("float32")
    )

    # ==================================================
    # SEARCH QUERY AFFINITY SCORE
    # ==================================================

    df["srch_query_affinity_score"] = (
        df["srch_query_affinity_score"]
        .fillna(
            stats[
                "srch_query_affinity_score_median"
            ]
        )
        .astype("float32")
    )

    # ==================================================
    # VISITOR HISTORY FEATURES
    # ==================================================

    df["visitor_hist_starrating"] = (
        df["visitor_hist_starrating"]
        .fillna(
            stats[
                "visitor_hist_starrating_median"
            ]
        )
        .astype("float32")
    )

    df["visitor_hist_adr_usd"] = (
        df["visitor_hist_adr_usd"]
        .fillna(
            stats[
                "visitor_hist_adr_usd_median"
            ]
        )
        .astype("float32")
    )

    # ==================================================
    # COMP RATE PERCENT DIFF FEATURES
    # ==================================================

    for i in range(1, 9):

        col = f"comp{i}_rate_percent_diff"

        df[col] = (
            df[col]
            .fillna(stats[f"{col}_median"])
            .astype("float32")
        )

    return df

In [ ]:
# Create LambdaMart Groups function
def create_lambdamart_groups(
    df,
    group_col="srch_id"
):

    groups = (
        df
        .groupby(group_col, observed=True)
        .size()
        .to_numpy()
    )

    return groups

In [ ]:
# Create Model Matrix Function
def create_model_matrix(
    df,
    drop_cols=DROP_COLS,
    target_col=TARGET_COL,
    has_target=True,
    return_metadata=False
):

    df = df.copy()

    # ==================================================
    # OPTIONAL METADATA
    # ==================================================

    metadata = df[[
        "srch_id",
        "prop_id"
    ]].copy()

    # ==================================================
    # CREATE X
    # ==================================================

    X = df.drop(
        columns=drop_cols,
        errors="ignore"
    )

    # ==================================================
    # CREATE y
    # ==================================================

    if has_target:

        y = df[target_col].copy()

        if return_metadata:
            return X, y, metadata

        return X, y

    # ==================================================
    # TEST SET
    # ==================================================

    if return_metadata:
        return X, metadata

    return X

### Load Datasets

In [28]:
#  ========== Load Training Data ==========
train_df = load_train_data(
    TRAIN_PATH,
    booking_relevance=5,
    click_relevance=1
)
print(f"Shape of training set: {train_df.shape}")
train_df.dtypes

Shape of training set: (4958347, 55)


srch_id                           int32
date_time                        object
site_id                           int32
visitor_location_country_id       int32
visitor_hist_starrating         float32
visitor_hist_adr_usd            float32
prop_country_id                   int32
prop_id                           int32
prop_starrating                    int8
prop_review_score               float32
prop_brand_bool                    bool
prop_location_score1            float32
prop_location_score2            float32
prop_log_historical_price       float32
position                           int8
price_usd                       float32
promotion_flag                     bool
srch_destination_id               int32
srch_length_of_stay                int8
srch_booking_window                int8
srch_adults_count                  int8
srch_children_count                int8
srch_room_count                    int8
srch_saturday_night_bool           bool
srch_query_affinity_score       float32


In [29]:
#  ========== Load Test Data ==========
test_df = load_test_data(TEST_PATH)
print(test_df.shape)
test_df.dtypes

(4959183, 50)


srch_id                           int32
date_time                        object
site_id                           int32
visitor_location_country_id       int32
visitor_hist_starrating         float32
visitor_hist_adr_usd            float32
prop_country_id                   int32
prop_id                           int32
prop_starrating                    int8
prop_review_score               float32
prop_brand_bool                    bool
prop_location_score1            float32
prop_location_score2            float32
prop_log_historical_price       float32
price_usd                       float32
promotion_flag                     bool
srch_destination_id               int32
srch_length_of_stay                int8
srch_booking_window                int8
srch_adults_count                  int8
srch_children_count                int8
srch_room_count                    int8
srch_saturday_night_bool           bool
srch_query_affinity_score       float32
orig_destination_distance       float32


# Part 1 - Preprocessing for Train, Validation, Test Sets

In [30]:
# ==================================================
# STATELESS PREPROCESSING
# ==================================================

train_df = stateless_preprocess(train_df)
test_df = stateless_preprocess(test_df)

# ==================================================
# TRAIN / VALIDATION SPLIT
# ==================================================

train_df, val_df = train_val_split(
    train_df,
    group_col="srch_id",
    test_size=TEST_SIZE,
    random_state=SEED
)

# ==================================================
# FIT LEARNED PREPROCESSING
# ==================================================

preprocessing_stats = fit_learned_preprocessing(
    train_df,
    price_usd_quantile=0.995
)

# ==================================================
# APPLY LEARNED PREPROCESSING
# ==================================================

train_df = apply_learned_preprocessing(
    train_df,
    preprocessing_stats
)

val_df = apply_learned_preprocessing(
    val_df,
    preprocessing_stats
)

test_df = apply_learned_preprocessing(
    test_df,
    preprocessing_stats
)

# ==================================================
# FEATURE ENGINEERING
# ==================================================

# train_df = add_features(train_df)
# val_df = add_features(val_df)
# test_df = add_features(test_df)


Train shape: (3966682, 70)
Validation shape: (991665, 70)
Test shape: (4959183, 65)


### Verifying Preprocessing

In [31]:
# ==================================================
# VERIFY SHAPES
# ==================================================

print("========== SHAPES ==========")

print(f"Train shape:      {train_df.shape}")
print(f"Validation shape: {val_df.shape}")
print(f"Test shape:       {test_df.shape}")

# ==================================================
# VERIFY REMAINING MISSING VALUES
# ==================================================

print("\n========== TRAIN NaNs ==========")

train_nans = (
    train_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(train_nans[train_nans > 0])

print("\n========== VALIDATION NaNs ==========")

val_nans = (
    val_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(val_nans[val_nans > 0])

print("\n========== TEST NaNs ==========")

test_nans = (
    test_df
    .isna()
    .sum()
    .sort_values(ascending=False)
)

print(test_nans[test_nans > 0])

# ==================================================
# VERIFY COLUMN CONSISTENCY
# ==================================================

print("\n========== COLUMN DIFFERENCES ==========")

train_only_cols = (
    set(train_df.columns)
    - set(test_df.columns)
)

test_only_cols = (
    set(test_df.columns)
    - set(train_df.columns)
)

print("\nColumns only in train/validation:")
print(sorted(train_only_cols))

print("\nColumns only in test:")
print(sorted(test_only_cols))

# ==================================================
# VERIFY DTYPES
# ==================================================

print("\n========== DTYPE SUMMARY ==========")

print(train_df.dtypes.value_counts())

# ==================================================
# VERIFY GROUP SPLIT INTEGRITY
# ==================================================

train_groups = set(train_df["srch_id"])
val_groups = set(val_df["srch_id"])

intersection = train_groups.intersection(val_groups)

print("\n========== GROUP SPLIT CHECK ==========")

print(f"Overlapping srch_id groups: {len(intersection)}")

if len(intersection) == 0:
    print("SUCCESS: No group leakage detected.")
else:
    print("WARNING: Group leakage detected.")

# ==================================================
# MEMORY USAGE
# ==================================================

print("\n========== MEMORY USAGE ==========")

train_mem = (
    train_df.memory_usage(deep=True).sum()
    / 1024**2
)

val_mem = (
    val_df.memory_usage(deep=True).sum()
    / 1024**2
)

test_mem = (
    test_df.memory_usage(deep=True).sum()
    / 1024**2
)

print(f"Train memory:      {train_mem:.2f} MB")
print(f"Validation memory: {val_mem:.2f} MB")
print(f"Test memory:       {test_mem:.2f} MB")

========== SHAPES ==========
Train shape:      (3966682, 70)
Validation shape: (991665, 70)
Test shape:       (4959183, 65)

========== TRAIN NaNs ==========
gross_bookings_usd    3855821
dtype: int64

========== VALIDATION NaNs ==========
gross_bookings_usd    964136
dtype: int64

========== TEST NaNs ==========
Series([], dtype: int64)

========== COLUMN DIFFERENCES ==========

Columns only in train/validation:
['booking_bool', 'click_bool', 'gross_bookings_usd', 'position', 'target']

Columns only in test:
[]

========== DTYPE SUMMARY ==========
bool              21
float32           18
category          16
int8               8
int32              6
datetime64[ns]     1
Name: count, dtype: int64

========== GROUP SPLIT CHECK ==========
Overlapping srch_id groups: 0
SUCCESS: No group leakage detected.

========== MEMORY USAGE ==========
Train memory:      593.92 MB
Validation memory: 148.48 MB
Test memory:       666.85 MB
